# Phase 11.2: SVM with an RBF kernel

**Owner:** Rotana · **Model:** `SVC` · **Writes to:** `data/processed/11_2_svm/`

**What it means:** An SVM finds the boundary with the widest gap between classes. The RBF kernel lets that boundary curve around groups of squares, which a straight line can't do.

**Key idea:** It can fit curved boundaries that softmax can't. The price is time, so the grid runs on a 10,000-square sample and only the winner is fitted on all of train.

Full step-by-step guide and the results sheet: [team plan](https://claude.ai/artifact/Rst8GHxAbeHavg4JS2RZV3) (tab 11.2).

### What we agreed on (same for all three notebooks)
- Train on the **46,408 balanced training squares**, score on the **36,000 validation squares**. The test set is not loaded.
- Pipeline: `StandardScaler` → `PCA(60, random_state=42)` → model.
- Tuning: `GridSearchCV` on train only, `GroupKFold(5)` grouped by photo, `scoring="f1_macro"`.
- Main metric: validation **macro F1** (floor to beat: **0.187**, always answering "none"). Second: **pothole (D40) recall**.

### What to tune

| Setting | Values to try | What it controls |
|---|---|---|
| `C` | 0.1, 1, 10, 100 | How hard it tries to get every training square right. |
| `gamma` | "scale", 0.01, 0.001 | How far one square's influence reaches. Large gamma gives wiggly boundaries. |

**Fixed:** `kernel="rbf"`, `class_weight="balanced"`; `probability=True` only on the final fit.

## 1. Setup

Everything shared lives in `ml/training.py`, so this notebook only adds the model:
- `load_data()`: the agreed rows (46,408 balanced training squares, 36,000 validation squares);
- `make_pipeline(clf)`: `StandardScaler` → `PCA(60)` → your model;
- `CV`: `GroupKFold(5)`, folds split by photo;
- `search_table(gs, OUT)`: saves the grid search to `cv_results.csv` and returns the best row;
- `report(...)`: the same evaluation for every model; writes `results.json` and `confusion_matrix.png`.

Don't change `ml/training.py` on your own: a change there changes all three results.

In [1]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV

sys.path.insert(0, str(Path.cwd().parents[1]))
from ml import config
from ml.training import CV, load_data, make_pipeline, output_dir, report, search_table

SEED = config.SEED
OUT = output_dir("11_2_svm")     # this notebook writes only here

data = load_data()                # balanced train + validation; the test set is never loaded
X_tr, y_tr, groups_tr, X_va, y_va = data
print("train", X_tr.shape, "| val", X_va.shape)

train (46408, 1769) | val (36000, 1769)


## 2. Baseline: is it working at all? (Phase 10)

Fit the model once with default settings and score it on validation.

**Check:** clearly above 0.187.

## 3. Grid search, on train only

Try every combination in the table above with `GridSearchCV(..., cv=CV, scoring="f1_macro", n_jobs=-1, return_train_score=True)` and `groups=groups_tr`.

**Check:** one row per combination in `gs.cv_results_`; note how long it took.

## 4. Read the search

`cv, best = search_table(gs, OUT)` saves the search and returns the best row. Plot the mean macro F1 per setting.

**Check:** if the best value is at the edge of the grid, add one value beyond it once and note it.

## 5. Final model, scored on validation

`report(pipe, data, OUT, "11.2 SVM", best, train_seconds)` writes `results.json` and `confusion_matrix.png`.

**Check:** read the D40 row of the confusion matrix: where do the missed potholes go?

## 6. Check the gap and the time

Compare train and validation macro F1, and write down how long the full fit took.

## 7. Diagnose before changing anything (Phase 12)

Train − validation gap above 0.15: overfitting. Both low and close: underfitting. Change **one** thing at a time, keep it only if validation macro F1 goes up, and log every try (also the failed ones).

## 8. Save and hand in

`joblib.dump(...)` the final pipeline to `OUT / "best_pipeline.joblib"`. Then import `results.json` in the Results sheet tab of the team plan, add your observations and download the PDF.